In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO("yolov8n-pose.pt")

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# COCO-style connections (YOLOv8 pose)
CONNECTIONS = [
    (5,7),(7,9),        # left arm
    (6,8),(8,10),       # right arm
    (5,6),              # shoulders
    (11,12),            # hips
    (5,11),(6,12),      # torso
    (11,13),(13,15),    # left leg
    (12,14),(14,16),    # right leg
    (0,5),(0,6)         # head to shoulders
]

def draw_skeleton(canvas, pts):
    # points
    for x, y in pts:
        if x > 0 and y > 0:
            cv2.circle(canvas, (int(x), int(y)), 4, (0,255,0), -1)
    # lines
    for i, j in CONNECTIONS:
        x1, y1 = pts[i]; x2, y2 = pts[j]
        if x1>0 and y1>0 and x2>0 and y2>0:
            cv2.line(canvas, (int(x1), int(y1)), (int(x2), int(y2)), (255,0,0), 2)

while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = model(frame, conf=0.25, verbose=False)

    # blank (black) right-panel
    skeleton = np.zeros_like(frame)

    kp = results[0].keypoints
    if kp is not None and hasattr(kp, "xy"):
        arr = kp.xy.cpu().numpy()   # shape: (num_people, 17, 2)
        if arr.shape[0] > 0:        # <-- important check
            for person_pts in arr:
                draw_skeleton(skeleton, person_pts)

    combined = np.hstack((frame, skeleton))
    cv2.imshow("You (Left)  vs  Skeleton (Right)", combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()